<center> <h1> 30-Day Readmission Risk Prediction </h1> </center>


<center> Alejandro Barajas, Sri Ramani Thungapati, Himanshu Singh Rao </center>


<h2>Introduction</h2>

In this report, we focused on the risk of a hospital patient being readmitted within 30 days. It’s essential to consider readmission because it can reflect the failure of initial treatments, help hospitals improve healthcare quality, reduce costs, and efficiently prepare the expected number of beds for people who are more than likely to be readmitted. To do this, we must implement a classification model because the primary goal is to classify patients who are at risk of readmission. 


<h2>High-level framework of solution:</h2>

To address this problem, our team followed four major stages of development. Stage 1 focused on building a strong foundation: each team member researched hospital readmission literature, conducted exploratory data analysis, and prepared the dataset for modeling. Stage 2 involved splitting the data into training, validation, and test sets, followed by implementing four baseline models—logistic regression, random forest, gradient boosting, and XGBoost. Based on their baseline performance, we selected logistic regression and XGBoost as the two models to tune and advance to Stage 3. Stage 3 consisted of optimizing these two models and comparing their performance to determine the stronger final model. Stage 4, the final stage, evaluated the models using classification metrics such as F1‑score, F2‑score, Brier score, ROC‑AUC, confusion matrix, and PR‑AUC. We then generated key visualizations and interpreted the results to draw meaningful conclusions.

<h2> Detailed aspects of the solution (implementation summary, important code snippets): </h2>

To perform this classification task, we used a dataset containing ... (data description). We first conducted an exploratory data analysis to identify any initial issues and confirmed that all columns had appropriate data types, with no missing or empty values. However, we identified three variables that were not relevant to our study. So readmission_risk_score, admission_date, and patient_id were removed from the dataset.

We then prepared the data by applying binary encoding to the gender variable and constructing separate preprocessing pipelines for numerical and categorical features. These were combined into a unified preprocessing pipeline in which numerical variables were standardized and categorical variables were one‑hot encoded. Finally, we performed a stratified train/validation/test split to ensure that the class imbalance in the target label was preserved across all subsets.

<h4> TASK: Baseline models implementation details and then followed by why logistic regression and XGBoost had the optimal potential for tuning: DO THIS IN THE TEXT BOX BELOW </h4> 

<h4> TASK: Tuned models implementation details and then followed by why logistic regression and XGBoost had the optimal potential for tuning: DO THIS IN THE TEXT BOX BELOW </h4> 

Optimized logistic regression model implementation details:

My group optimized the logistic regression model using GridSearchCV and capacity‑based thresholding. To begin, we tuned the model’s hyperparameters by defining a parameter grid that included C, logistic_l1_ratio, logistic_solver, and logistic_class_weight. The parameter C controls the strength of regularization, where smaller values impose stronger regularization. The l1_ratio determines the type of penalty applied: a value of 0 corresponds to ridge (L2), a value of 1 corresponds to lasso (L1), and intermediate values represent elastic net. The solver specifies the optimization algorithm used to minimize the logistic regression loss; we selected saga because it is the only solver that supports elastic net regularization through l1_ratio. Finally, class_weight adjusts how much the model penalizes errors for each class. Because our target label is highly imbalanced, we set class_weight to “balanced,” so the model gives more weight to the minority class.

For the grid search, we used average precision (PR‑AUC) as the scoring metric, since the dataset is imbalanced and our task places a higher value on precision. Below is a representation of our group’s workflow.

In [ ]:
param_grid = {
    "logistic__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "logistic__l1_ratio": [0, 0.25, 0.5, 0.75, 1],   # 0 = L2, 1 = L1
    "logistic__solver": ["saga"],   # saga is the only solver supporting l1_ratio
    "logistic__class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="average_precision", # Class label is highly imbalanced so this is the optimal scoring metric
    cv=5,
    n_jobs = -1,
    verbose=0
)

The next step was to tune the classification threshold on the validation set using a capacity‑based threshold after completing the initial training on the training set. To apply this method, we selected the best‑performing model from our GridSearchCV results and chose a capacity level of 60%. This means the model was allowed to flag at most 60% of patients as high‑risk. We selected this value because, in a realistic clinical setting, flagging a much higher proportion would lead to over‑classification and overwhelm available resources. A 60% capacity provides a practical balance: it allows the model to identify a substantial portion of potentially high‑risk patients without producing an operationally unrealistic number of alerts. Below is the implementation of this approach.

In [ ]:
# Capacity‑based threshold tuning (top‑K) 
best_model = grid.best_estimator_
capacity = 0.60  # flag top 60%
# Get validation probabilities
y_val_probs = best_model.predict_proba(X_val)[:, 1]
# Compute threshold at the (1 - capacity) percentile
optimal_threshold = np.percentile(y_val_probs, 100 * (1 - capacity))

Lastly now we can evaluate the model on the test set...

<b>Second optimized model implementation details:</b> Write here

<h4> TASK: explain why we chose the final model in the box below</h4>

<h2> Results and analysis (evaluate results, compare with reference or baseline solutions, what worked what didn’t work, and compare the two models we created)</h2>

<h2> Conclusion</h2>

<h2> Future work </h2>